In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/navjotkaushal/human-vs-ai-generated-essays/balanced_ai_human_prompts.csv'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [17]:
import pandas as pd 
df=pd.read_csv('/kaggle/input/datasets/navjotkaushal/human-vs-ai-generated-essays/balanced_ai_human_prompts.csv')
df['text']=df['text'].str.lower()

In [19]:
import re 
def remove_url(text):
    text=re.sub(r'https\S+','',text)
    return text 
df['text']=df['text'].apply(remove_url)

In [21]:
def remove_punctuation(text):
    text=re.sub(r'[^A-Za-z0-9\s]','',text)
    return text 
df['text']=df['text'].apply(remove_punctuation)

In [22]:
def remove_html(text):
    text=re.sub(r'<.*?>','',text)
    return text 
df['text']=df['text'].apply(remove_html)

In [24]:

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


def remove_stopwords(text):
    token=word_tokenize(text)
    stop_w=stopwords.words('english')
    for words in token:
        if words in stop_w:
            text=text.replace(words,'')
    return text 
df['text']=df['text'].apply(remove_stopwords)

In [26]:
from nltk.stem import PorterStemmer 
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    token=word_tokenize(text)
    for tokens in token:
        stemmed_words.append(ps.stem(tokens))
    return ' '.join(stemmed_words)  
df['text']=df['text'].apply(stemming)

In [28]:
from sklearn.preprocessing import LabelEncoder 
le=LabelEncoder()
df['generated']=le.fit_transform(df['generated'])
y=df['generated']

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df['text'])

In [31]:
from sklearn.model_selection import train_test_split 
import torch 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
from torch.utils.data import TensorDataset,DataLoader
x_train=x_train.toarray()
x_test=x_test.toarray()
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float(),  
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float(),  
)
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [32]:
import torch
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])
        return out

In [33]:
import torch.optim as optim 
input_layer=x_train.shape[1]
model=RNN(input_layer)
criterion=nn.BCELoss()
optimiser=optim.Adam(model.parameters())

In [34]:
best_val_loss = float("inf")
nums_of_epoch=10
for epoch in range(nums_of_epoch):
    model.train()
    for xb, yb in train_loader:
        optimiser.zero_grad()
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        probs = torch.sigmoid(outputs.squeeze())
        loss = criterion(probs, yb)
        loss.backward()
        optimiser.step()
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.unsqueeze(1)
            outputs = model(xb)
            probs = torch.sigmoid(outputs.squeeze())
            loss = criterion(probs, yb)
            val_loss += loss.item()
    val_loss /= len(test_loader)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rnn_weights.pth")

    print(f"epoch {epoch+1} | val_loss={val_loss:.4f} | best={best_val_loss:.4f}")


epoch 1 | val_loss=0.2000 | best=0.2000
epoch 2 | val_loss=0.0534 | best=0.0534
epoch 3 | val_loss=0.0303 | best=0.0303
epoch 4 | val_loss=0.0258 | best=0.0258
epoch 5 | val_loss=0.0223 | best=0.0223
epoch 6 | val_loss=0.0217 | best=0.0217
epoch 7 | val_loss=0.0206 | best=0.0206
epoch 8 | val_loss=0.0203 | best=0.0203
epoch 9 | val_loss=0.0251 | best=0.0203
epoch 10 | val_loss=0.0199 | best=0.0199


In [35]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)
